In [1]:
!git clone https://github.com/AMAAI-Lab/text2midi
#!cd text2midi
!pip install -r text2midi/requirements.txt

fatal: destination path 'text2midi' already exists and is not an empty directory.
  Using cached torch-2.5.1-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached triton-3.1.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 44.9 MB/s eta 0:00:00
Using cached torch-2.5.1-cp311-cp311-manylinux1_x86_64.whl (906.5 MB)
Using cached triton-3.1.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (209.5 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.3.1
    Uninstalling triton-3.3.1:
      Successfully uninstalled triton-3.3.1
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.3
    Uninstalling sympy-1.13.3:
      Successfully uninstalled sympy-1.13.3
  Attempting uninstall: torch
    Found existing installation: torch 2.7.1+cu118
    Uninstalling torch-2.7.1+cu118:
      Successfully uninstalled torch-2.7.1+cu118
ERROR: pip's dependency

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p "/content/drive/MyDrive/slp_text2midi/text2midi_custom_model"

In [10]:
!cd text2midi

In [3]:
!pip install --upgrade torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/triton-3.3.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.5 kB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp311-cp311-manylinux_2_28_x86_64.whl (905.3 MB)
Using cached https://download.pytorch.org/whl/triton-3.3.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (155.7 MB)
Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl (6.2 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.1.0
    Uninstalling triton-3.1.0:
      Successfully uninstalled triton-3.1.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalli

In [4]:
!pip install music21 librosa pretty_midi scipy

In [ ]:
%load_ext autoreload
%autoreload 2

In [12]:
# Cell 0: Mount drive & set path for custom MoE files
import sys
sys.path.append("/content/text2midi")  # or wherever you cloned the repo

sys.path.append("/content/drive/MyDrive/slp_text2midi/text2midi_custom_model")

# Cell 1: Import libraries and MoE model
import os, pickle, torch, numpy as np
import torch.nn as nn
from transformers import T5Tokenizer
from huggingface_hub import hf_hub_download
import soundfile as sf
#from transformer_with_moe import TransformerWithMoE  # MoE version
import torch
import torch.nn as nn

#//////////////////////////////////////////////////////////
from model.transformer_model import Transformer
import torch.nn.functional as F

class MoEFeedForward(nn.Module):
    def __init__(self, input_dim, hidden_dim=2048, num_experts=4, dropout=0.1):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
                nn.Dropout(dropout)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        gate_logits = self.gate(x)
        gate_weights = F.softmax(gate_logits, dim=-1)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)
        return torch.sum(expert_outputs * gate_weights.unsqueeze(2), dim=-1)


#from moe_feedforward import MoEFeedForward
class TransformerWithMoE(nn.Module):
    def __init__(self, vocab_size, device='cpu'):
        super().__init__()
        self.core = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
        self.moe = MoEFeedForward(input_dim=768, hidden_dim=2048, num_experts=4)
        self.output_proj = nn.Linear(768, vocab_size)

    #1def forward(self, input_ids, decoder_input_ids=None, attention_mask=None):
        #core_out = self.core(input_ids, decoder_input_ids, attention_mask)
        #moe_out = self.moe(core_out)
        #logits = self.output_proj(moe_out)
        #return logits
    #2def forward(self, input_ids, decoder_input_ids=None, attention_mask=None):
    #    core_out = self.core(input_ids, decoder_input_ids, attention_mask)

        # Fix shape if core returns logits directly
    #    if core_out.shape[-1] == self.output_proj.out_features:
    #        core_out = torch.matmul(core_out, self.output_proj.weight)  # project vocab_size → hidden_dim

    #    moe_out = self.moe(core_out)
    #    logits = self.output_proj(moe_out)
    #    return logits

    def forward(self, input_ids, decoder_input_ids=None, attention_mask=None):
          core_out = self.core(input_ids, decoder_input_ids, attention_mask)

          # If output is already logits (vocab size), skip MoE + projection
          if core_out.shape[-1] == self.output_proj.out_features:
              return core_out  # already logits

          # Otherwise, go through MoE + projection
          if core_out.shape[-1] != self.output_proj.in_features:
              raise ValueError(f"Expected core output last dim to match {self.output_proj.in_features}, got {core_out.shape}")

          moe_out = self.moe(core_out)
          return self.output_proj(moe_out)




    def generate(self, input_ids, attention_mask=None, max_len=2000, temperature=1.0):
        return self.core.generate(input_ids, attention_mask, max_len=max_len, temperature=temperature)

#//////////////////////////////////////////////////////////

# Ensure you're in the cloned repo directory
#os.chdir('text2midi')

# Cell 2: Download pretrained weights
repo_id = "amaai-lab/text2midi"
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Cell 3: Load tokenizer & MoE-based model
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

vocab_size = len(r_tokenizer)
print("Vocab size:", vocab_size)

model = TransformerWithMoE(vocab_size, device=device)
model.core.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
print("MoE-enhanced model loaded.")

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)

In [ ]:
#from transformer_with_moe import TransformerWithMoE
model = TransformerWithMoE(vocab_size, device=device)
model.core.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

NameError: name 'TransformerWithMoE' is not defined

In [5]:
!pip install music21 librosa pretty_midi scipy

In [ ]:
!pip install laion-clap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 8.9 MB/s eta 0:00:00
  Created wheel for progressbar: filename=progressbar-2.5-py3-none-any.whl size=12065 sha256=0694b22c670387f178008ba7dd8a3544e9951db3aadc537069c56e403946c03c
  Stored in directory: /root/.cache/pip/wheels/8d/bb/b2/5353b966ac6f3c5e1000629a9a5f6aed41794487f551e32efc
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=94d7c664bd6fa10f2476d64524d707672182715e720c733468267dbeb6757f2c
  Stored in directory: /root/.cache/pip/wheels/40/b3/0f/a40dbd1c6861731779f62cc4babcb234387e11d697df70ee97
Successful

In [12]:
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer
#from model.transformer_model import Transformer # Now, the import should work
from huggingface_hub import hf_hub_download
import music21
import numpy as np
from scipy.spatial.distance import cosine
import librosa;
import pretty_midi
from typing import List, Dict, Tuple
import json
import random
from datasets import load_dataset
import tempfile
import requests

# CLAP imports - you may need to install these
#try:
#  import laion_clap
#except ImportError:
#    print("Warning: laion_clap not installed. Install with: pip install laion-clap")
#    laion_clap = None

In [ ]:
def safe_float_conversion(value):
    """Safely convert numpy scalars or other types to Python float"""
    if isinstance(value, (np.floating, np.integer)):
        return float(value)
    elif isinstance(value, (int, float)):
        return float(value)
    elif hasattr(value, 'item'):  # numpy scalar
        return float(value.item())
    else:
        try:
            return float(value)
        except:
            return 0.0


# Cell 4: Define ObjectiveEvaluator Class
class ObjectiveEvaluator:
    def __init__(self):
        # Initialize CLAP model if available
        self.clap_model = None
        if laion_clap is not None:
            try:
                self.clap_model = laion_clap.CLAP_Module(enable_fusion=False)
                self.clap_model.load_ckpt() # Load the default checkpoint
                print("CLAP model loaded successfully")
            except Exception as e:
                print(f"Failed to load CLAP model: {e}")

        # Tempo bins as defined in the paper
        self.tempo_bins = [40, 60, 70, 90, 110, 140, 160, 210]

    def evaluate_single_sample(self, text: str, generated_midi_path: str, ground_truth_midi_path: str) -> Dict:
        """Evaluate a single sample with all metrics - returns Python native types"""
        results = {}

        # Compression Ratio
        results['compression_ratio_generated'] = safe_float_conversion(
            self.compute_compression_ratio(generated_midi_path)
        )
        results['compression_ratio_ground_truth'] = safe_float_conversion(
            self.compute_compression_ratio(ground_truth_midi_path)
        )

        # CLAP Similarity - this is where the error was occurring
        clap_sim = self.compute_clap_similarity(text, generated_midi_path)
        results['clap_similarity'] = safe_float_conversion(clap_sim)

        # Feature-wise comparison
        gen_tempo = self.extract_tempo(generated_midi_path)
        gt_tempo = self.extract_tempo(ground_truth_midi_path)
        results['generated_tempo'] = safe_float_conversion(gen_tempo)
        results['ground_truth_tempo'] = safe_float_conversion(gt_tempo)

        tb, tbt = self.compute_tempo_bin_accuracy(gen_tempo, gt_tempo)
        results['tempo_bin_match'] = bool(tb)  # Ensure boolean
        results['tempo_bin_tolerance_match'] = bool(tbt)  # Ensure boolean

        gen_key = self.extract_key(generated_midi_path)
        gt_key = self.extract_key(ground_truth_midi_path)
        results['generated_key'] = str(gen_key)  # Ensure string
        results['ground_truth_key'] = str(gt_key)  # Ensure string

        ck, ckd = self.compute_key_accuracy(gen_key, gt_key)
        results['correct_key'] = bool(ck)  # Ensure boolean
        results['correct_key_duplicates'] = bool(ckd)  # Ensure boolean

        return results

    def midi_to_audio(self, midi_file_path: str, sample_rate: int = 22050) -> np.ndarray:
        """Convert MIDI file to audio using pretty_midi and fluidsynth"""
        try:
            # Load MIDI file
            midi_data = pretty_midi.PrettyMIDI(midi_file_path)

            # Check if MIDI has any instruments
            if len(midi_data.instruments) == 0:
                print("Warning: MIDI file has no instruments, returning silence")
                return np.zeros(sample_rate * 2, dtype=np.float32)  # 2 seconds of silence

            # Synthesize audio
            audio = midi_data.synthesize(fs=sample_rate)

            # **FIX: Ensure we always return a proper numpy array**
            if not isinstance(audio, np.ndarray):
                audio = np.array(audio, dtype=np.float32)
            else:
                audio = audio.astype(np.float32)

            # Handle edge cases - ensure we always return an array, never a scalar
            if audio.size == 0:
                print("Warning: MIDI synthesis produced empty audio, returning silence")
                return np.zeros(sample_rate * 2, dtype=np.float32)
            elif audio.size == 1:
                print("Warning: MIDI synthesis produced single sample, extending to array")
                # Create an array by repeating the single value
                single_value = float(audio.item()) if hasattr(audio, 'item') else float(audio)
                return np.full(sample_rate * 2, single_value, dtype=np.float32)
            elif np.isscalar(audio):
                # Handle scalar case explicitly
                print("Warning: MIDI synthesis returned scalar, converting to array")
                return np.full(sample_rate * 2, float(audio), dtype=np.float32)

            # Ensure it's 1D
            if audio.ndim > 1:
                audio = audio.flatten()
            elif audio.ndim == 0:  # Handle 0-dimensional arrays
                single_value = float(audio.item()) if hasattr(audio, 'item') else float(audio)
                return np.full(sample_rate * 2, single_value, dtype=np.float32)

            return audio

        except Exception as e:
            print(f"Error converting MIDI to audio: {e}")
            # Return silence instead of empty array
            return np.zeros(22050 * 2, dtype=np.float32)  # 2 seconds of silence at 22050 Hz

    def compute_clap_similarity(self, text: str, midi_file_path: str) -> float:
        """Compute CLAP similarity between text and MIDI audio"""
        if self.clap_model is None:
            print("CLAP model not available")
            return 0.0

        try:
            # Convert MIDI to audio
            audio = self.midi_to_audio(midi_file_path)

            # **FIX: Additional validation to ensure we have a proper array**
            if not isinstance(audio, np.ndarray):
                print(f"Error: midi_to_audio returned {type(audio)} instead of np.ndarray")
                return 0.0

            if audio.ndim == 0 or np.isscalar(audio):
                print("Error: audio is scalar, converting to array")
                audio = np.full(22050, float(audio) if np.isscalar(audio) else float(audio.item()), dtype=np.float32)

            if audio.size == 0:
                print("Error: audio array is empty")
                return 0.0

            # Ensure minimum length for CLAP processing
            if len(audio) < 1000:
                if len(audio) == 0:
                    audio = np.zeros(22050, dtype=np.float32)  # 1 second of silence
                else:
                    repeat_factor = max(1, 1000 // len(audio) + 1)
                    audio = np.tile(audio, repeat_factor)[:22050]  # Ensure reasonable length

            print(f"Final audio validation - type: {type(audio)}, shape: {audio.shape}, dtype: {audio.dtype}")

            # Get embeddings
            text_embed = self.clap_model.get_text_embedding([text], use_tensor=False)

            # **FIX: CLAP library workaround - save audio to temporary file**
            # The CLAP library has internal issues with numpy arrays, so we'll use file-based approach
            import tempfile
            import scipy.io.wavfile as wavfile

            try:
                # Create temporary wav file
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp_file:
                    temp_path = tmp_file.name

                # Normalize audio to 16-bit range and save as WAV
                audio_normalized = audio.astype(np.float64)
                # Ensure audio is in proper range [-1, 1]
                if np.max(np.abs(audio_normalized)) > 0:
                    audio_normalized = audio_normalized / np.max(np.abs(audio_normalized))

                # Convert to 16-bit integer
                audio_16bit = (audio_normalized * 32767).astype(np.int16)

                # Save as WAV file
                wavfile.write(temp_path, 22050, audio_16bit)

                print(f"Saved temporary audio file: {temp_path}")

                # Try file-based CLAP embedding methods
                audio_embed = None

                # Method 1: Try get_audio_embedding_from_filelist if available
                if hasattr(self.clap_model, 'get_audio_embedding_from_filelist'):
                    try:
                        audio_embed = self.clap_model.get_audio_embedding_from_filelist([temp_path], use_tensor=False)
                        print("Successfully used get_audio_embedding_from_filelist")
                    except Exception as e:
                        print(f"get_audio_embedding_from_filelist failed: {e}")

                # Method 2: Try get_audio_embedding if available
                if audio_embed is None and hasattr(self.clap_model, 'get_audio_embedding'):
                    try:
                        audio_embed = self.clap_model.get_audio_embedding([temp_path], use_tensor=False)
                        print("Successfully used get_audio_embedding")
                    except Exception as e:
                        print(f"get_audio_embedding failed: {e}")

                # Method 3: Load the file back and try data method with fresh array
                if audio_embed is None:
                    try:
                        # Load the audio file we just saved
                        sample_rate, audio_from_file = wavfile.read(temp_path)

                        # Convert back to float
                        audio_from_file = audio_from_file.astype(np.float64) / 32767.0

                        # Ensure it's a proper numpy array
                        audio_from_file = np.asarray(audio_from_file, dtype=np.float64)

                        print(f"Loaded audio from file - type: {type(audio_from_file)}, shape: {audio_from_file.shape}, dtype: {audio_from_file.dtype}")

                        audio_embed = self.clap_model.get_audio_embedding_from_data(x=audio_from_file, use_tensor=False)
                        print("Successfully used get_audio_embedding_from_data with file-loaded audio")

                    except Exception as e:
                        print(f"File-based data method failed: {e}")

                # Clean up temporary file
                import os
                if os.path.exists(temp_path):
                    os.unlink(temp_path)

                if audio_embed is None:
                    print("All CLAP embedding methods failed, returning 0.0")
                    return 0.0

            except Exception as e:
                print(f"Error in file-based CLAP approach: {e}")
                return 0.0

            # Unwrap embeddings if they come as lists
            if isinstance(text_embed, list):
                text_embed = text_embed[0]
            if isinstance(audio_embed, list):
                audio_embed = audio_embed[0]

            # Convert to numpy arrays if they aren't already
            if not isinstance(text_embed, np.ndarray):
                text_embed = np.array(text_embed)
            if not isinstance(audio_embed, np.ndarray):
                audio_embed = np.array(audio_embed)

            # Ensure they are proper numpy arrays with float64 dtype
            text_embed = np.asarray(text_embed, dtype=np.float64)
            audio_embed = np.asarray(audio_embed, dtype=np.float64)

            # Handle different dimensionalities
            if text_embed.ndim > 1:
                text_embed = text_embed.flatten()
            if audio_embed.ndim > 1:
                audio_embed = audio_embed.flatten()

            # Final validation
            if not isinstance(text_embed, np.ndarray) or not isinstance(audio_embed, np.ndarray):
                print(f"CLAP embeddings are not ndarray: text {type(text_embed)}, audio {type(audio_embed)}")
                return 0.0

            if text_embed.size == 0 or audio_embed.size == 0:
                print("CLAP embeddings are empty")
                return 0.0

            if text_embed.shape != audio_embed.shape:
                print(f"CLAP embedding shape mismatch: text {text_embed.shape}, audio {audio_embed.shape}")
                return 0.0

            # Compute cosine similarity using manual calculation
            # Normalize embeddings
            text_embed_norm = text_embed / (np.linalg.norm(text_embed) + 1e-8)
            audio_embed_norm = audio_embed / (np.linalg.norm(audio_embed) + 1e-8)

            # Compute cosine similarity manually
            similarity = np.dot(text_embed_norm, audio_embed_norm)

            # Ensure we return a Python float, not numpy scalar
            if hasattr(similarity, 'item'):
                return float(similarity.item())
            else:
                return float(similarity)

        except Exception as e:
            print(f"Error computing CLAP similarity: {e}")
            import traceback
            traceback.print_exc()  # This will help debug the exact error
            return 0.0

    def compute_compression_ratio(self, midi_file_path: str) -> float:
        """
        Compute compression ratio using COSIATEC algorithm approximation
        Note: This is a simplified version. For exact COSIATEC, you'd need the original implementation
        """
        try:
            # Load MIDI file
            score = music21.converter.parse(midi_file_path)

            # Extract notes and their properties
            notes = []
            for element in score.recurse().notes:
                if hasattr(element, 'pitch'):  # Single note
                    notes.append({
                        'pitch': element.pitch.midi,
                        'offset': float(element.offset),
                        'duration': float(element.duration.quarterLength)
                    })
                elif hasattr(element, 'pitches'):  # Chord
                    for pitch in element.pitches:
                        notes.append({
                            'pitch': pitch.midi,
                            'offset': float(element.offset),
                            'duration': float(element.duration.quarterLength)
                        })

            if len(notes) == 0:
                return 0.0

            # Simple pattern detection based on repeated pitch sequences
            # This is a simplified approximation of COSIATEC
            patterns = {}
            window_size = 4  # Look for patterns of 4 notes

            for i in range(len(notes) - window_size + 1):
                pattern = tuple(note['pitch'] for note in notes[i:i+window_size])
                if pattern in patterns:
                    patterns[pattern] += 1
                else:
                    patterns[pattern] = 1

            # Calculate compression ratio
            total_notes = len(notes)
            repeated_notes = sum(count * len(pattern) for pattern, count in patterns.items() if count > 1)
            compression_ratio = repeated_notes / total_notes if total_notes > 0 else 0.0

            return compression_ratio

        except Exception as e:
            print(f"Error computing compression ratio: {e}")
            return 0.0

    def extract_tempo(self, midi_file_path: str) -> float:
        """Extract tempo from MIDI file"""
        try:
            score = music21.converter.parse(midi_file_path)
            tempos = score.metronomeMarkBoundaries()
            if tempos:
                return float(tempos[0][2].number)
            else:
                # Fallback: analyze note density
                notes = list(score.recurse().notes)
                if len(notes) > 1:
                    duration = float(score.duration.quarterLength)
                    note_density = len(notes) / duration
                    estimated_tempo = note_density * 60  # Rough estimation
                    return max(60, min(200, estimated_tempo))  # Clamp to reasonable range
                return 120  # Default tempo
        except Exception as e:
            print(f"Error extracting tempo: {e}")
            return 120

    def extract_key(self, midi_file_path: str) -> str:
        """Extract key from MIDI file"""
        try:
            score = music21.converter.parse(midi_file_path)
            key = score.analyze('key')
            return str(key)
        except Exception as e:
            print(f"Error extracting key: {e}")
            return "C major"

    def get_tempo_bin(self, tempo: float) -> int:
        """Get the tempo bin index for a given tempo"""
        for i, bin_edge in enumerate(self.tempo_bins):
            if tempo < bin_edge:
                return i
        return len(self.tempo_bins)

    def compute_tempo_bin_accuracy(self, generated_tempo: float, ground_truth_tempo: float) -> tuple:
        """Compute TB and TBT metrics"""
        gen_bin = self.get_tempo_bin(generated_tempo)
        gt_bin = self.get_tempo_bin(ground_truth_tempo)

        # TB: exact bin match
        tb = (gen_bin == gt_bin)

        # TBT: exact bin or neighboring bin
        tbt = abs(gen_bin - gt_bin) <= 1

        return tb, tbt

    def normalize_key(self, key_str: str) -> str:
        """Normalize key string for comparison"""
        try:
            key = music21.key.Key(key_str)
            return str(key).lower().strip()
        except:
            return key_str.lower().strip()

    def are_relative_keys(self, key1: str, key2: str) -> bool:
        """Check if two keys are relative (major and its relative minor)"""
        try:
            k1 = music21.key.Key(key1)
            k2 = music21.key.Key(key2)

            if k1.mode == 'major' and k2.mode == 'minor':
                return k1.relative.name == k2.name
            elif k1.mode == 'minor' and k2.mode == 'major':
                return k1.relative.name == k2.name
            else:
                return k1.name == k2.name
        except:
            return False

    def compute_key_accuracy(self, generated_key: str, ground_truth_key: str) -> tuple:
        """Compute CK and CKD metrics"""
        # CK: exact key match
        ck = self.normalize_key(generated_key) == self.normalize_key(ground_truth_key)

        # CKD: exact key or relative key match
        ckd = ck or self.are_relative_keys(generated_key, ground_truth_key)

        return ck, ckd

In [7]:
#vol 2
from datasets import load_dataset
from typing import List, Dict
import os
import random
import json
import numpy as np

def load_midicaps_samples(num_samples: int = 100, split: str = "train", midi_dir: str = None) -> List[Dict]:
    """
    Load captions from MidiCaps (Hugging Face) and match with local MIDI files from lmd_full
    """
    if midi_dir is None:
        print("⚠️ You must provide the path to your local lmd_full directory.")
        return []

    print(f"Using MIDI directory: {midi_dir}")
    print("Loading MidiCaps dataset from Hugging Face...")

    try:
        dataset = load_dataset("amaai-lab/MidiCaps", split=split)
        print(f"✅ Loaded {len(dataset)} samples from '{split}' split.")

        if num_samples > len(dataset):
            print("⚠️ Requested more samples than available. Using all.")
            num_samples = len(dataset)

        selected_indices = random.sample(range(len(dataset)), num_samples)
        samples = []

        for i, idx in enumerate(selected_indices):
            sample = dataset[idx]
            caption = sample['caption']
            midi_rel_path = sample['location']  # e.g. 'lmd_full/2/abc123.mid'

            # Strip leading 'lmd_full/' if present
            if midi_rel_path.startswith("lmd_full/"):
                midi_rel_path = midi_rel_path[len("lmd_full/"):]

            midi_abs_path = os.path.join(midi_dir, midi_rel_path)

            if not os.path.exists(midi_abs_path):
                print(f"❌ Missing MIDI file: {midi_abs_path}")
                continue

            samples.append({
                'text': caption,
                'midi_path': midi_abs_path,
                'original_index': idx,
                'sample_id': i
            })

        print(f"✅ Collected {len(samples)} valid samples with existing local MIDI files.")
        return samples

    except Exception as e:
        print(f"❌ Error loading MidiCaps dataset: {e}")
        return []


def batch_evaluate_midicaps(evaluator, model, tokenizer, r_tokenizer, device, num_samples: int = 100, midi_dir: str = None):
    """
    Evaluate a batch of MidiCaps samples using the specified model and evaluator.
    """
    test_samples = load_midicaps_samples(num_samples=num_samples, midi_dir=midi_dir)
    if not test_samples:
        print("❌ No valid test samples found.")
        return []

    all_results = []

    for i, sample in enumerate(test_samples):
        print(f"\nProcessing sample {i+1}/{len(test_samples)} (Original index: {sample['original_index']})")

        text = sample['text']
        gt_midi_path = sample['midi_path']
        print(f"Caption: {text[:100]}...")
        print(f"Ground truth MIDI: {gt_midi_path}")

        if not os.path.exists(gt_midi_path):
            print(f"❌ MIDI file not found: {gt_midi_path}")
            continue

        try:
            # Tokenize caption
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
            input_ids = inputs.input_ids.to(device)
            attention_mask = inputs.attention_mask.to(device)

            # Generate MIDI token ids
            with torch.no_grad():
                output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_len=128)
            output_list = output[0].tolist()

            # Decode MIDI
            generated_midi = r_tokenizer.decode(output_list)
            generated_path = f"generated_{i}.mid"
            generated_midi.dump_midi(generated_path)

            # Evaluate
            results = evaluator.evaluate_single_sample(text, generated_path, gt_midi_path)

            # Clean and store results
            results_cleaned = {k: (safe_float_conversion(v) if not isinstance(v, (str, bool)) else v) for k, v in results.items()}
            results_cleaned.update({
                'sample_id': i,
                'original_index': sample['original_index'],
                'caption': text,
                'ground_truth_path': gt_midi_path
            })
            all_results.append(results_cleaned)

            os.remove(generated_path)

        except Exception as e:
            print(f"❌ Error evaluating sample {i}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if not all_results:
        print("⚠️ No samples were successfully evaluated.")
        return []

    # Aggregate results
    print(f"\n=== AGGREGATE RESULTS ({len(all_results)} samples) ===")
    compression_gen = [safe_float_conversion(r['compression_ratio_generated']) for r in all_results]
    compression_gt = [safe_float_conversion(r['compression_ratio_ground_truth']) for r in all_results]
    print(f"Average Compression Ratio (Gen): {np.mean(compression_gen):.4f}")
    print(f"Average Compression Ratio (GT): {np.mean(compression_gt):.4f}")

    clap_sims = [safe_float_conversion(r['clap_similarity']) for r in all_results if safe_float_conversion(r['clap_similarity']) > 0]
    if clap_sims:
        print(f"Average CLAP Similarity: {np.mean(clap_sims):.4f} ± {np.std(clap_sims):.4f}")
    else:
        print("No valid CLAP similarities computed")

    tb_acc = np.mean([float(r['tempo_bin_match']) for r in all_results])
    tbt_acc = np.mean([float(r['tempo_bin_tolerance_match']) for r in all_results])
    ck_acc = np.mean([float(r['correct_key']) for r in all_results])
    ckd_acc = np.mean([float(r['correct_key_duplicates']) for r in all_results])
    print(f"Tempo Bin Accuracy (TB): {tb_acc:.4f} | TBT: {tbt_acc:.4f}")
    print(f"Correct Key Accuracy (CK): {ck_acc:.4f} | CKD: {ckd_acc:.4f}")

    # Save to disk
    results_file = f"evaluation_results_model_{len(all_results)}_samples.json"
    with open(results_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"📁 Results saved to {results_file}")

    return all_results


# **Evaluate Without Retraining (Zero-shot)**

In [ ]:
evaluator = ObjectiveEvaluator()
midi_dir = "/content/drive/MyDrive/slp_text2midi/midicaps/lmd_full"
results = batch_evaluate_midicaps(evaluator, num_samples=100, midi_dir=midi_dir)

# **miniTraining**

In [ ]:
# Example: load 10000 MIDI-caption pairs from MidiCaps
midi_dir = "/content/drive/MyDrive/slp_text2midi/midicaps/lmd_full"  # ✅ Adjust to your actual path
samples = load_midicaps_samples(num_samples=10000, midi_dir=midi_dir)


Using MIDI directory: /content/drive/MyDrive/slp_text2midi/midicaps/lmd_full
Loading MidiCaps dataset from Hugging Face...


README.md:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/428M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/168385 [00:00<?, ? examples/s]

✅ Loaded 168385 samples from 'train' split.
✅ Collected 10000 valid samples with existing local MIDI files.


In [14]:
os.chdir('text2midi')

In [15]:
# === PART 1: Your Model Code - FIXED ===

import os, pickle, torch, numpy as np
import torch.nn as nn
from transformers import T5Tokenizer
from huggingface_hub import hf_hub_download
import soundfile as sf
import torch
import torch.nn as nn
from model.transformer_model import Transformer
import torch.nn.functional as F

class MoEFeedForward(nn.Module):
    def __init__(self, input_dim, hidden_dim=2048, num_experts=4, dropout=0.1):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
                nn.Dropout(dropout)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        gate_logits = self.gate(x)
        gate_weights = F.softmax(gate_logits, dim=-1)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)
        return torch.sum(expert_outputs * gate_weights.unsqueeze(2), dim=-1)

class TransformerWithMoE(nn.Module):
    def __init__(self, vocab_size, device='cpu'):
        super().__init__()
        self.core = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
        self.moe = MoEFeedForward(input_dim=768, hidden_dim=2048, num_experts=4)
        self.output_proj = nn.Linear(768, vocab_size)

        # ✅ FIX: Store hidden states from core
        self.last_hidden_states = None

    def forward(self, input_ids, decoder_input_ids=None, attention_mask=None):
        # ✅ FIX: Modified forward pass to handle gradients properly

        # Register hook to capture hidden states before final projection
        def capture_hidden_hook(module, input, output):
            # Look for the right shape - we want hidden states (batch, seq, 768)
            if isinstance(output, torch.Tensor) and len(output.shape) == 3 and output.shape[-1] == 768:
                self.last_hidden_states = output

        # Find and hook the right layer
        hook_handle = None
        for name, module in self.core.named_modules():
            # Hook to layer norm or final transformer layer
            if any(keyword in name.lower() for keyword in ['layer_norm', 'ln_f', 'norm']) and isinstance(module, nn.LayerNorm):
                hook_handle = module.register_forward_hook(capture_hidden_hook)
                break

        try:
            # Forward through core
            core_out = self.core(input_ids, decoder_input_ids, attention_mask)

            # Use captured hidden states if available
            if self.last_hidden_states is not None:
                hidden_states = self.last_hidden_states
            else:
                # Fallback: if core returns logits, convert to hidden states
                if core_out.shape[-1] == self.output_proj.out_features:
                    # ✅ FIX: Proper gradient handling
                    # Use a learned projection instead of matrix multiplication
                    if not hasattr(self, 'reverse_proj'):
                        self.reverse_proj = nn.Linear(self.output_proj.out_features, 768).to(core_out.device)
                        # Initialize reverse projection
                        with torch.no_grad():
                            self.reverse_proj.weight.data = self.output_proj.weight.data.T

                    hidden_states = self.reverse_proj(core_out)
                else:
                    hidden_states = core_out

            # ✅ FIX: Ensure gradients flow
            if not hidden_states.requires_grad:
                hidden_states = hidden_states.requires_grad_(True)

            # Apply MoE
            moe_out = self.moe(hidden_states)

            # Final projection
            logits = self.output_proj(moe_out)
            return logits

        finally:
            # Clean up hook
            if hook_handle is not None:
                hook_handle.remove()
            self.last_hidden_states = None

    def generate(self, input_ids, attention_mask=None, max_len=2000, temperature=1.0):
        return self.core.generate(input_ids, attention_mask, max_len=max_len, temperature=temperature)


In [16]:
# === PART 2: Your Dataset Code - UNCHANGED ===

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import torch

class MidiCaptionDataset(Dataset):
    def __init__(self, data, midi_tokenizer, caption_tokenizer, max_len=1024):
        self.data = data
        self.midi_tokenizer = midi_tokenizer         # Miditok tokenizer
        self.caption_tokenizer = caption_tokenizer   # T5 tokenizer
        self.max_len = max_len
        self.pad_id = getattr(midi_tokenizer, 'pad_token_id', 0) or 0

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Correctly extract caption and MIDI path
        caption = item.get('caption') or item.get('text') or ""
        midi_path = item.get('midi') or item.get('midi_path')

        # Tokenize caption (T5 tokenizer)
        input_tokens = self.caption_tokenizer(
            caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=128
        )
        input_ids = input_tokens.input_ids.squeeze()
        attention_mask = input_tokens.attention_mask.squeeze()

        # Tokenize MIDI (Miditok tokenizer)
        midi_seq = self.midi_tokenizer.encode(midi_path)
        midi_ids = midi_seq.ids[:128]  # must match model output sequence length
        midi_ids += [self.pad_id] * (128 - len(midi_ids))

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(midi_ids)
        }

In [ ]:
# === PART 3: Your Training Code - FIXED ===

def train_one_epoch(model, dataloader, optimizer, pad_id, device):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)           # T5 input (caption)
        attention_mask = batch['attention_mask'].to(device) # T5 attention
        labels = batch['labels'].to(device)                 # Target MIDI tokens

        optimizer.zero_grad()

        try:
            # ✅ Forward pass — only input_ids and attention_mask
            logits = model(input_ids=input_ids, attention_mask=attention_mask)

            # ✅ Check shape: (batch_size, seq_len, vocab_size) vs (batch_size, seq_len)
            if logits.shape[:2] != labels.shape:
                print(f"Warning: Shape mismatch: logits {logits.shape}, labels {labels.shape}")
                # Try to fix by truncating or padding
                min_seq_len = min(logits.shape[1], labels.shape[1])
                logits = logits[:, :min_seq_len, :]
                labels = labels[:, :min_seq_len]

            # Flatten for loss
            logits = logits.view(-1, logits.shape[-1])
            labels = labels.view(-1)

            # Compute loss and backprop
            loss = nn.CrossEntropyLoss(ignore_index=pad_id)(logits, labels)

            # ✅ FIX: Check for NaN loss
            if torch.isnan(loss):
                print("Warning: NaN loss detected, skipping batch")
                continue

            loss.backward()

            # ✅ FIX: Add gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        except Exception as e:
            print(f"Error in training step: {e}")
            continue

    return total_loss / len(dataloader)

In [ ]:
# === PART 4: Your Setup Code - MODIFIED FOR BETTER FREEZING ===

# Download pretrained weights (your code)
repo_id = "amaai-lab/text2midi"
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer & MoE-based model (your code)
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

vocab_size = len(r_tokenizer)
print("Vocab size:", vocab_size)

model = TransformerWithMoE(vocab_size, device=device)
model.core.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
print("MoE-enhanced model loaded.")


Using device: cuda
Vocab size: 524
MoE-enhanced model loaded.


In [ ]:
# ✅ MODIFIED: Better parameter freezing strategy
print("Setting up parameter freezing...")

# Don't freeze the ENTIRE core - be more selective
trainable_count = 0
frozen_count = 0

for name, param in model.named_parameters():
    if 'moe' in name or 'output_proj' in name:
        # Always train MoE components
        param.requires_grad = True
        trainable_count += 1
        print(f"Trainable: {name}")
    elif 'core' in name:
        # For core: freeze early layers, keep some top layers trainable
        if any(layer in name for layer in ['embed', 'layer.0', 'layer.1', 'layer.2']):
            param.requires_grad = False
            frozen_count += 1
        else:
            param.requires_grad = True
            trainable_count += 1
            print(f"Trainable (core): {name}")
    else:
        param.requires_grad = True
        trainable_count += 1

print(f"Total trainable parameters: {trainable_count}")
print(f"Total frozen parameters: {frozen_count}")

# ✅ FIXED: Create dataset with correct argument order
dataset = MidiCaptionDataset(samples, r_tokenizer, tokenizer)
train_loader = DataLoader(dataset, batch_size=2, shuffle=True)

# ✅ MODIFIED: Better optimizer with parameter groups
core_params = [p for n, p in model.named_parameters() if 'core' in n and p.requires_grad]
moe_params = [p for n, p in model.named_parameters() if ('moe' in n or 'output_proj' in n) and p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': core_params, 'lr': 1e-5, 'weight_decay': 0.01},      # Lower LR for core
    {'params': moe_params, 'lr': 2e-4, 'weight_decay': 0.01}        # Higher LR for MoE
])

# Training loop (your code structure, but with fixes)
pad_id = getattr(r_tokenizer, 'pad_token_id', 0) or 0

print("Starting training...")
for epoch in range(3):
    print(f"\nEpoch {epoch+1}/3")
    avg_loss = train_one_epoch(model, train_loader, optimizer, pad_id, device)
    print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
        'vocab_size': vocab_size
    }
    torch.save(checkpoint, f"moe_finetuned_epoch{epoch+1}.pt")
    print(f"Saved checkpoint: moe_finetuned_epoch{epoch+1}.pt")

print("Training completed!")

# ✅ BONUS: Function to load your trained model later
def load_trained_moe_model(checkpoint_path, vocab_size, device):
    """Load your trained MoE model from checkpoint"""
    model = TransformerWithMoE(vocab_size, device=device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    model.to(device)
    model.eval()

    print(f"Loaded model from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
    return model


Setting up parameter freezing...
Trainable (core): core.input_emb.weight
Trainable (core): core.pos_encoder.pe
Trainable (core): core.encoder.shared.weight
Trainable (core): core.encoder.encoder.final_layer_norm.weight
Trainable (core): core.decoder.layers.0.self_attn.to_qkv.weight
Trainable (core): core.decoder.layers.0.self_attn.to_out.weight
Trainable (core): core.decoder.layers.0.multihead_attn.in_proj_weight
Trainable (core): core.decoder.layers.0.multihead_attn.in_proj_bias
Trainable (core): core.decoder.layers.0.multihead_attn.out_proj.weight
Trainable (core): core.decoder.layers.0.multihead_attn.out_proj.bias
Trainable (core): core.decoder.layers.0.linear1.weight
Trainable (core): core.decoder.layers.0.linear1.bias
Trainable (core): core.decoder.layers.0.linear2.weight
Trainable (core): core.decoder.layers.0.linear2.bias
Trainable (core): core.decoder.layers.0.norm1.weight
Trainable (core): core.decoder.layers.0.norm1.bias
Trainable (core): core.decoder.layers.0.norm2.weight
Tr

Training: 100%|██████████| 5000/5000 [2:36:10<00:00,  1.87s/it]


Epoch 1 - Loss: 4.9361
Saved checkpoint: moe_finetuned_epoch1.pt

Epoch 2/3


Training: 100%|██████████| 5000/5000 [17:41<00:00,  4.71it/s]


Epoch 2 - Loss: 4.8924
Saved checkpoint: moe_finetuned_epoch2.pt

Epoch 3/3


Training: 100%|██████████| 5000/5000 [17:53<00:00,  4.66it/s]


Epoch 3 - Loss: 4.8833
Saved checkpoint: moe_finetuned_epoch3.pt
Training completed!


In [ ]:
# Load finetuned MoE model
checkpoint_path = "moe_finetuned_epoch3.pt"  # Or any other checkpoint
trained_model = load_trained_moe_model(checkpoint_path, vocab_size, device)


Loaded model from epoch 3 with loss 4.8833


In [17]:
import os
import torch
import pickle
from transformers import T5Tokenizer

from huggingface_hub import hf_hub_download

vocab_path = hf_hub_download(repo_id="amaai-lab/text2midi", filename="vocab_remi.pkl")

# === Paths ===epoch3.p
checkpoint_path = "/content/drive/MyDrive/slp_text2midi/text2midi_checkpoints/moe_finetuned_epoch3.pt"
midi_dir = "/content/drive/MyDrive/slp_text2midi/midicaps/lmd_full"  # Local MIDI root
num_eval_samples = 100

# === Device setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load tokenizer ===
with open(vocab_path, "rb") as f:
    r_tokenizer = pickle.load(f)
vocab_size = len(r_tokenizer)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

# === Load MoE model ===
def load_trained_moe_model(checkpoint_path, vocab_size, device):
    model = TransformerWithMoE(vocab_size, device=device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f"✅ Loaded model from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
    return model

trained_model = load_trained_moe_model(checkpoint_path, vocab_size, device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab_remi.pkl:   0%|          | 0.00/27.3k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

✅ Loaded model from epoch 3 with loss 4.8833


In [19]:
gate_weights = trained_model.moe.gate.weight  # shape: (num_experts, input_dim)
gate_bias = trained_model.moe.gate.bias       # shape: (num_experts,)
print(f"Gate weights shape: {gate_weights.shape}")
print(f"Gate bias shape: {gate_bias.shape}")

Gate weights shape: torch.Size([4, 768])
Gate bias shape: torch.Size([4])


In [21]:
for i, expert in enumerate(trained_model.moe.experts):
    print(f"Expert {i} weights:")
    print(expert[0].weight)  # First Linear layer
    print(expert[2].weight)  # Second Linear layer


Expert 0 weights:
Parameter containing:
tensor([[-6.1466e-03,  1.1166e-02, -1.7099e-02,  ..., -2.6519e-02,
         -3.5848e-02,  1.9466e-03],
        [-7.1532e-03, -2.2740e-05,  1.9720e-02,  ...,  1.8702e-02,
         -7.2710e-03,  2.8641e-02],
        [ 3.3339e-02,  1.3416e-02,  3.6338e-02,  ...,  3.2933e-02,
         -9.7531e-05,  4.7399e-03],
        ...,
        [ 3.8200e-02, -6.9177e-03,  1.9770e-07,  ...,  3.7882e-02,
         -1.0042e-02, -1.5984e-02],
        [-6.0238e-03, -1.3159e-02, -2.8668e-02,  ...,  2.6886e-02,
          1.1971e-02,  3.8551e-02],
        [-2.8113e-02, -1.8167e-02, -2.8712e-02,  ..., -1.0251e-02,
         -2.4746e-02,  2.1396e-02]], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([[ 0.0003,  0.0073, -0.0075,  ...,  0.0021,  0.0169, -0.0041],
        [ 0.0171, -0.0204,  0.0133,  ...,  0.0071,  0.0045,  0.0073],
        [ 0.0076, -0.0130, -0.0218,  ...,  0.0170,  0.0077, -0.0172],
        ...,
        [-0.0208,  0.0181, -0.0179,  ..., -0.0

In [22]:
class MoEFeedForward(nn.Module):
    def __init__(self, input_dim, hidden_dim=2048, num_experts=4, dropout=0.1):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
                nn.Dropout(dropout)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x, return_gates=False):
        gate_logits = self.gate(x)                          # (B, T, num_experts)
        gate_weights = F.softmax(gate_logits, dim=-1)       # (B, T, num_experts)

        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)  # (B, T, D, E)
        moe_output = torch.sum(expert_outputs * gate_weights.unsqueeze(2), dim=-1)    # (B, T, D)

        if return_gates:
            return moe_output, gate_weights
        return moe_output


In [23]:
import os
import torch
import pickle
from transformers import T5Tokenizer

from huggingface_hub import hf_hub_download

vocab_path = hf_hub_download(repo_id="amaai-lab/text2midi", filename="vocab_remi.pkl")

# === Paths ===epoch3.p
checkpoint_path = "/content/drive/MyDrive/slp_text2midi/text2midi_checkpoints/moe_finetuned_epoch3.pt"
midi_dir = "/content/drive/MyDrive/slp_text2midi/midicaps/lmd_full"  # Local MIDI root
num_eval_samples = 100

# === Device setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load tokenizer ===
with open(vocab_path, "rb") as f:
    r_tokenizer = pickle.load(f)
vocab_size = len(r_tokenizer)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

# === Load MoE model ===
def load_trained_moe_model(checkpoint_path, vocab_size, device):
    model = TransformerWithMoE(vocab_size, device=device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f"✅ Loaded model from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
    return model

trained_model = load_trained_moe_model(checkpoint_path, vocab_size, device)


✅ Loaded model from epoch 3 with loss 4.8833


In [28]:

from transformers import T5Tokenizer

# Load T5 tokenizer
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

# Example caption
caption = "A cheerful pop song with a touch of electronic elements, featuring a piano and drums. The melody evokes feelings of love and happiness, with a hint of Christmas spirit. Set in the key of D major and a moderate 4/4 tempo, the song follows a chord progression of A7, D, Em7, A7, and D throughout its duration."

# Tokenize
tokens = tokenizer(
    caption,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = tokens.input_ids.squeeze()
attention_mask = tokens.attention_mask.squeeze()

In [33]:
import torch.nn as nn

# Manually create and attach reverse_proj
trained_model.reverse_proj = nn.Linear(
    trained_model.output_proj.out_features,
    trained_model.output_proj.in_features
).to(device)

# Copy transposed weights
with torch.no_grad():
    trained_model.reverse_proj.weight.copy_(trained_model.output_proj.weight.T)


In [34]:
if input_ids.dim() == 1:
    input_ids = input_ids.unsqueeze(0)
if attention_mask.dim() == 1:
    attention_mask = attention_mask.unsqueeze(0)
if decoder_input_ids.dim() == 1:
    decoder_input_ids = decoder_input_ids.unsqueeze(0)

# Move inputs to the correct device
input_ids = input_ids.to(device)
decoder_input_ids = torch.zeros_like(input_ids)
attention_mask = attention_mask.to(device)



# Run model.core to extract hidden representations (without updating weights)
with torch.no_grad():
    core_out = trained_model.core(input_ids,decoder_input_ids,attention_mask)

    # If the core output is logits, convert it back to hidden states
    if core_out.shape[-1] == trained_model.output_proj.out_features:
        core_out = trained_model.reverse_proj(core_out)

    # Pass through MoE, collect both output and gate weights
    moe_out, gate_weights = trained_model.moe(core_out, return_gates=True)  # gate_weights: (batch_size, seq_len, num_experts)

    # Compute average usage across batch and sequence
    avg_usage = gate_weights.mean(dim=[0, 1])  # shape: (num_experts,)
    print("Average expert usage:", avg_usage.cpu().numpy())


Average expert usage: [0.26499748 0.03847584 0.00229722 0.6942294 ]


In [ ]:
evaluator = ObjectiveEvaluator()
results = batch_evaluate_midicaps(
    evaluator=evaluator,
    model=trained_model,
    tokenizer=tokenizer,
    r_tokenizer=r_tokenizer,
    device=device,
    num_samples=100,
    midi_dir="/content/drive/MyDrive/slp_text2midi/midicaps/lmd_full"
)

/usr/local/lib/python3.11/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Load our best checkpoint in the paper.
Download completed!
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Loaded
aud

100%|██████████| 128/128 [00:05<00:00, 21.97it/s]


...

Final audio validation - type: <class 'numpy.ndarray'>, shape: (110220,), dtype: float32
Saved temporary audio file: /tmp/tmphxm2hdcs.wav
Successfully used get_audio_embedding_from_filelist

Processing sample 99/100 (Original index: 97952)
Caption: This upbeat rock and pop fusion track radiates happiness and energy, perfect for corporate or motiva...
Ground truth MIDI: /content/drive/MyDrive/slp_text2midi/midicaps/lmd_full/b/bd97c17f44024e7fd614ccb67e38465c.mid


100%|██████████| 128/128 [00:06<00:00, 19.64it/s]


Final audio validation - type: <class 'numpy.ndarray'>, shape: (58657,), dtype: float32
Saved temporary audio file: /tmp/tmp0npczv6o.wav
Successfully used get_audio_embedding_from_filelist

Processing sample 100/100 (Original index: 158122)
Caption: A dark and melodic rock and electronic song featuring piano, overdriven guitar, drums, fretless bass...
Ground truth MIDI: /content/drive/MyDrive/slp_text2midi/midicaps/lmd_full/6/643c6cda5149fd5137f5a23fe79964b6.mid


100%|██████████| 128/128 [00:06<00:00, 19.01it/s]


Final audio validation - type: <class 'numpy.ndarray'>, shape: (98404,), dtype: float32
Saved temporary audio file: /tmp/tmp9tkc_1qk.wav
Successfully used get_audio_embedding_from_filelist

=== AGGREGATE RESULTS (100 samples) ===
Average Compression Ratio (Gen): 0.1086
Average Compression Ratio (GT): 2.8975
Average CLAP Similarity: 0.1351 ± 0.0899
Tempo Bin Accuracy (TB): 0.3600 | TBT: 0.7200
Correct Key Accuracy (CK): 0.1800 | CKD: 0.1800
📁 Results saved to evaluation_results_model_100_samples.json
